# Nested Monte Carlo Power Simulator for the Region-as-Subject Cluster-Permutation Test

Sizes the *real* noise-sweep study for cluster-permutation (see
`pilot_noise_cluster_permutation.ipynb` for the test itself, run on the existing
26-region/4-rep/5-level pilot). Cluster-permutation doesn't reduce to a closed-form
power formula the way the ANOVA-style tests elsewhere in this project do, so this
is genuinely nested Monte Carlo: simulate a synthetic pilot dataset under an assumed
ground truth, run the *actual* cluster-permutation procedure on it (its own internal
permutation resampling included), repeat many times, and see how often the known
true effect is detected. Repeat that across a grid of candidate `(k regions, R
repetitions/level, S SNR levels)` designs.

**Denser SNR grid, same range**: candidate designs add log-spaced points *between*
the existing `[20, 10, 5, 2, 1]` (kept exactly), not extending the range -- e.g. a
9-level candidate is `[1, 1.5, 2, 3, 5, 7, 10, 14, 20]`.

**Ground truth for the simulation comes from the existing pilot, not invented
numbers**: per `(outcome, competitor)`, the observed 26x5 region-by-SNR paired
difference matrix (`wCSET - competitor`, sign-adjusted, averaged over the 4 existing
repetitions -- see the companion notebook) gives:
- `true_mean_diff(snr)`: the population-average difference at each of the 5
  existing SNR levels, log-linearly interpolated to any new candidate SNR value.
- `sigma2_region_diff(snr)`: genuine between-region variance in that difference,
  recovered via method-of-moments (observed between-region variance minus the
  *known* noise floor `2 * sigma2_resid(snr) / R_pilot` -- the `2x` because a
  paired difference of two methods' idiosyncratic residual noise adds variance from
  both sides; floored at 0). Checked against the real data first: the noise floor
  is well below the observed variance for every outcome/competitor pair checked, so
  this decomposition isn't degenerate.
- `sigma2_resid(snr)`: reused directly from the existing per-level fits
  (`pilot_noise_variance_components.json`), no refitting.

**This notebook is written to be run later, not now** -- the grid/iteration counts
below are set modestly so a smoke-test subset is cheap, but the full sweep is real
compute. Reduce `N_OUTER`/the candidate grids for a quick check, restore them for
the real run.

In [1]:
import json
import numpy as np
import pandas as pd
from scipy import stats

OUTCOMES = ["localization_error_hrp", "ap_hrp", "ap_pp", "correlation_auc"]
HIGHER_IS_BETTER = {"localization_error_hrp": False, "ap_hrp": True, "ap_pp": True, "correlation_auc": True}
REFERENCE_METHOD = "wcset"
COMPETITORS = ["cset", "mce", "sloreta", "dspm", "mne"]
SNR_EXISTING = [20.0, 10.0, 5.0, 2.0, 1.0]
R_PILOT = 4

df = pd.read_csv("pilot_noise_outcomes.tsv", sep="\t")
vc = json.load(open("pilot_noise_variance_components.json"))
regions = sorted(df["region"].unique())
k_pilot = len(regions)
print(f"{len(df)} rows, {k_pilot} regions (pilot), existing snr levels {SNR_EXISTING}")

3120 rows, 26 regions (pilot), existing snr levels [20.0, 10.0, 5.0, 2.0, 1.0]


## Ground truth per (outcome, competitor): mean difference + region variance vs. SNR

Fit once from the existing pilot data, then interpolated (in `log(SNR)`) to
whatever candidate SNR grid a given design point uses.

In [2]:
def region_series(outcome, competitor):
    """(k_pilot, 5) matrix: per-region, per-existing-SNR-level mean paired difference
    (wcset - competitor), sign-adjusted so positive = wcset better, averaged over the
    4 pilot repetitions."""
    sign = 1.0 if HIGHER_IS_BETTER[outcome] else -1.0
    wide_wcset = df[df["method"] == REFERENCE_METHOD].groupby(["region", "snr"])[outcome].mean().unstack()[SNR_EXISTING]
    wide_comp = df[df["method"] == competitor].groupby(["region", "snr"])[outcome].mean().unstack()[SNR_EXISTING]
    diff = sign * (wide_wcset - wide_comp)
    return diff.reindex(regions).to_numpy()


def fit_ground_truth(outcome, competitor):
    D = region_series(outcome, competitor)
    sigma2_resid_existing = np.array([vc[str(int(s))][outcome]["sigma2_resid"] for s in SNR_EXISTING])
    noise_floor = 2 * sigma2_resid_existing / R_PILOT
    observed_var = D.var(axis=0, ddof=1)
    sigma2_region_diff_existing = np.maximum(0.0, observed_var - noise_floor)
    return {
        "log_snr_existing": np.log(SNR_EXISTING),
        "true_mean_diff_existing": D.mean(axis=0),
        "sigma2_region_diff_existing": sigma2_region_diff_existing,
        "sigma2_resid_existing": sigma2_resid_existing,
    }


def interp_ground_truth(gt, snr_values):
    x = np.log(snr_values)
    # np.interp needs increasing xp -- log_snr_existing is decreasing (20->1), so flip
    xp = gt["log_snr_existing"][::-1]
    true_mean = np.interp(x, xp, gt["true_mean_diff_existing"][::-1])
    sigma2_region = np.interp(x, xp, gt["sigma2_region_diff_existing"][::-1])
    sigma2_resid = np.interp(x, xp, gt["sigma2_resid_existing"][::-1])
    return true_mean, sigma2_region, sigma2_resid


ground_truths = {(o, c): fit_ground_truth(o, c) for o in OUTCOMES for c in COMPETITORS}
print("Ground truth fit for", len(ground_truths), "(outcome, competitor) pairs.")

Ground truth fit for 20 (outcome, competitor) pairs.


## Cluster-permutation test (unchanged from the companion notebook)

In [3]:
def cluster_permutation_test(D, cluster_alpha=0.05, n_perm=500, seed=0):
    """D: (k, S) array, one series per region (subject). Region-level permutation:
    flips each row's sign consistently across all S columns."""
    k, S = D.shape
    rng = np.random.default_rng(seed)
    t_crit = stats.t.ppf(1 - cluster_alpha / 2, df=k - 1)

    def compute_t(Dperm):
        m = Dperm.mean(axis=0)
        se = Dperm.std(axis=0, ddof=1) / np.sqrt(k)
        return m / se

    def find_clusters(t):
        clusters = []
        sign_mask = np.where(t > t_crit, 1, np.where(t < -t_crit, -1, 0))
        i = 0
        while i < S:
            if sign_mask[i] != 0:
                j = i
                while j < S and sign_mask[j] == sign_mask[i]:
                    j += 1
                clusters.append((i, j - 1, np.sum(t[i:j]), sign_mask[i]))
                i = j
            else:
                i += 1
        return clusters

    t_obs = compute_t(D)
    clusters_obs = find_clusters(t_obs)

    max_pos_null = np.zeros(n_perm)
    for p in range(n_perm):
        flips = rng.choice([-1.0, 1.0], size=k)
        t_p = compute_t(D * flips[:, None])
        pos = [c[2] for c in find_clusters(t_p) if c[3] == 1]
        max_pos_null[p] = max(pos) if pos else 0.0

    results = []
    for (i, j, mass, sgn) in clusters_obs:
        if sgn == 1:
            pval = (np.sum(max_pos_null >= mass) + 1) / (n_perm + 1)
            results.append({"i": i, "j": j, "mass": mass, "p_value": pval})
    return results

## Simulate one synthetic pilot dataset for a candidate (k, R, snr_values) design

In [4]:
def simulate_region_series(true_mean, sigma2_region, sigma2_resid, k, R, rng):
    """(k, S) synthetic paired-difference matrix. `true_mean`/`sigma2_region`/
    `sigma2_resid` are (S,) arrays already interpolated to the candidate snr grid."""
    S = len(true_mean)
    z = rng.normal(0, 1, size=k)                                   # one draw per region ("subject")
    region_effect = z[:, None] * np.sqrt(sigma2_region)[None, :]    # (k, S) -- same region, scaled per level
    noise = rng.normal(0, 1, size=(k, S)) * np.sqrt(2 * sigma2_resid / R)[None, :]
    return true_mean[None, :] + region_effect + noise


def true_cluster_mask(true_mean, min_effect_frac=0.1):
    """Which candidate-grid SNR levels count as "really" part of the effect, for
    scoring a hit -- any level where the true mean is at least min_effect_frac of the
    largest true mean (avoids requiring detection of near-zero-effect edges)."""
    return true_mean >= min_effect_frac * true_mean.max()

## Nested power sweep

For each candidate `(k, R, snr_values)` design and each `(outcome, competitor)`:
simulate `N_OUTER` synthetic datasets, run cluster-permutation on each (smaller
inner `n_perm` -- a power estimate doesn't need publication-grade per-run
precision), and score a "hit" as finding a significant positive cluster that
overlaps the true-effect region. Power = hit rate.

**Cost warning**: `N_OUTER x n_designs x n_(outcome,competitor) x inner permutations`
-- the values below are deliberately small for a smoke test. Restore
`N_OUTER=300`, a fuller `DESIGN_GRID`, and `INNER_N_PERM=500` for the real run.

In [5]:
N_OUTER = 20            # smoke-test default -- raise to ~200-300 for the real run
INNER_N_PERM = 100       # smoke-test default -- raise to ~500 for the real run
SEED = 0

DESIGN_GRID = [
    {"k": 20, "R": 3, "snr_values": [1, 1.5, 2, 3, 5, 7, 10, 14, 20]},
    {"k": 26, "R": 4, "snr_values": [20, 10, 5, 2, 1]},   # the already-run pilot, as a sanity-check baseline
]

# smoke test: just one representative (outcome, competitor) pair, the hardest one
# (localization error was the marginal outcome in the real cluster-permutation run)
SMOKE_TEST_PAIRS = [("localization_error_hrp", "cset")]

rng = np.random.default_rng(SEED)
power_rows = []

for design in DESIGN_GRID:
    k, R, snr_values = design["k"], design["R"], design["snr_values"]
    for (outcome, competitor) in SMOKE_TEST_PAIRS:
        gt = ground_truths[(outcome, competitor)]
        true_mean, sigma2_region, sigma2_resid = interp_ground_truth(gt, snr_values)
        true_mask = true_cluster_mask(true_mean)

        hits = 0
        for _ in range(N_OUTER):
            D = simulate_region_series(true_mean, sigma2_region, sigma2_resid, k, R, rng)
            clusters = cluster_permutation_test(D, n_perm=INNER_N_PERM, seed=rng.integers(1e9))
            hit = any(true_mask[c["i"]:c["j"] + 1].any() and c["p_value"] < 0.05 for c in clusters)
            hits += hit

        power_rows.append({
            "k": k, "R": R, "S": len(snr_values), "outcome": outcome, "competitor": competitor,
            "power": hits / N_OUTER,
        })
        print(f"k={k}, R={R}, S={len(snr_values)}, {outcome} vs {competitor}: power={hits/N_OUTER:.2f}")

power_df = pd.DataFrame(power_rows)
power_df

k=20, R=3, S=9, localization_error_hrp vs cset: power=0.95
k=26, R=4, S=5, localization_error_hrp vs cset: power=1.00


,k,R,S,outcome,competitor,power
0,20,3,9,localization_error_hrp,cset,0.95
1,26,4,5,localization_error_hrp,cset,1.00


## Running the real sweep

Restore full iteration counts and expand `SMOKE_TEST_PAIRS` to the outcomes/
competitors you actually want sized (localization error is the outcome most worth
spending compute on -- AP/correlation were already saturated at the original
pilot's scale) before running this for real. Expand `DESIGN_GRID` with more
`(k, R, S)` combinations to actually find the smallest adequate design rather than
just comparing two candidates.